<a href="https://colab.research.google.com/github/SalvadorGR23/Tesis/blob/main/METODOLOG%C3%8DA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**IMPORTACIÓN DE LIBRERÍAS**

In [96]:
!pip install PyPortfolioOpt

import yfinance as yf
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

from datetime import date
from pypfopt import EfficientFrontier, plotting

pd.options.display.float_format = '{:,.6f}'.format
np.random.seed(23)

#**DEFINICIÓN DE CORTES DE VOLATILIDAD POR PERFIL DE INVERSIONISTA**

##**LÍMITES DE INVERSIÓN EN RENTA VARIABLE POR PERFIL DE INVERSIONISTA**

In [97]:
limites_inversion = {
  'Agresivo Superior': {'IPC': 1, 'CETES': 0},
  'Agresivo Inferior': {'IPC': 0.45, 'CETES': 0.55},
  'Moderado Superior': {'IPC': 0.45, 'CETES': 0.55},
  'Moderado Inferior': {'IPC': 0.10, 'CETES': 0.90},
  'Conservador Superior': {'IPC': 0.10, 'CETES': 0.9},
  'Conservador Inferior': {'IPC': 0, 'CETES': 1}
}

In [98]:
IPC = yf.Ticker('^MXX').history(start = '2025-01-01', end = '2026-01-01')['Close']
IPC = IPC.dropna(how = 'all')
IPC = IPC.reset_index()
IPC['Date'] = pd.to_datetime(IPC['Date']).dt.date
IPC

,Date,Close
0,2025-01-02,"49,765.199219"
1,2025-01-03,"48,957.238281"
2,2025-01-06,"49,493.558594"
3,2025-01-07,"50,085.500000"
4,2025-01-08,"49,634.261719"
...,...,...
246,2025-12-24,"65,616.429688"
247,2025-12-26,"65,636.359375"
248,2025-12-29,"65,347.078125"
249,2025-12-30,"64,366.699219"


In [99]:
varianza_ipc = IPC['Close'].pct_change().var()

print('Varianza IPC: ', round(varianza_ipc, 4))

Varianza IPC:  0.0001


In [100]:
perfiles_riesgo = []
varianzas_agresivo = []
varianzas_moderado = []
varianzas_conservador = []

for perfil, pesos in limites_inversion.items():
  peso_ipc = pesos['IPC']

  varianza_perfil = (peso_ipc*peso_ipc)*varianza_ipc

  if 'Agresivo' in perfil:
    perfiles_riesgo.append('Perfil Agresivo')
    varianzas_agresivo.append(round(varianza_perfil, 8))
  elif 'Moderado' in perfil:
    perfiles_riesgo.append('Perfil Moderado')
    varianzas_moderado.append(round(varianza_perfil, 8))
  elif 'Conservador' in perfil:
    perfiles_riesgo.append('Perfil Conservador')
    varianzas_conservador.append(round(varianza_perfil, 8))

perfiles_riesgo = ['Perfil Agresivo', 'Perfil Moderado', 'Perfil Conservador']
varianzas_agresivo = sorted(varianzas_agresivo)
varianzas_moderado = sorted(varianzas_moderado)
varianzas_conservador = sorted(varianzas_conservador)

varianzas_minimas = [varianzas_agresivo[0], varianzas_moderado[0], varianzas_conservador[0]]
varianzas_maximas = [varianzas_agresivo[-1], varianzas_moderado[-1], varianzas_conservador[-1]]

CORTES_VOLATILIDADES_PERFILES_DE_RIESGO = pd.DataFrame({
    'Perfiles de Riesgo': perfiles_riesgo,
    'Varianza Mínima': varianzas_minimas,
    'Varianza Máxima': varianzas_maximas
})

CORTES_VOLATILIDADES_PERFILES_DE_RIESGO

,Perfiles de Riesgo,Varianza Mínima,Varianza Máxima
0,Perfil Agresivo,0.000020,0.000097
1,Perfil Moderado,0.000001,0.000020
2,Perfil Conservador,0.000000,0.000001


In [101]:
varianzas_conservador

[0.0, 9.7e-07]

#**ASIGNACIÓN DE ACCIONES POR PERFIL DE INVERSIONISTA**

##**EMPRESAS LISTADAS EN LA BOLSA MEXICANA DE VALORES**
Se presenta un diccionario con las empresas listadas en la BMV al 31 de diciembre de 2025 y posteriormente se crea un dataframe con los mismos datos.

In [102]:
empresas_bmv = {
  'VISTAA.MX': ['VISTA ENERGY, S.A.B. DE C.V.', 'ENERGÍA'],
  'CIEB.MX': ['CORPORACION INTERAMERICANA DE ENTRETENIMIENTO, S.A.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'CHDRAUIB.MX': ['GRUPO COMERCIAL CHEDRAUI, S.A.B. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'CTAXTELA.MX': ['CONTROLADORA AXTEL, S.A.B. DE C.V.', 'SERVICIOS DE TELECOMUNICACIONES'],
  'PASAB.MX': ['PROMOTORA AMBIENTAL, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'BEVIDESB.MX': ['FARMACIAS BENAVIDES, S.A.B. DE C.V.', 'SALUD'],
  'FRAGUAB.MX': ['CORPORATIVO FRAGUA, S.A.B. DE C.V.', 'SALUD'],
  'GENTERA.MX': ['GENTERA, S.A.B. DE C.V.', 'SERVICIOS FINANCIEROS'],
  'ACTINVRB.MX': ['CORPORACION ACTINVER, S.A.B. DE C.V.', 'SERVICIOS FINANCIEROS'],
  'Q.MX': ['QUÁLITAS CONTROLADORA, S.A.B. DE C.V.', 'SERVICIOS FINANCIEROS'],
  'TMMA.MX': ['GRUPO TMM, S.A.', 'INDUSTRIAL'],
  'GCARSOA1.MX': ['GRUPO CARSO, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'PINFRAL.MX': ['PROMOTORA Y OPERADORA DE INFRAESTRUCTURA, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'GFINBURO.MX': ['GRUPO FINANCIERO INBURSA, S.A.B. DE C.V.', 'SERVICIOS FINANCIEROS'],
  'GPROFUT.MX': ['GRUPO PROFUTURO, S.A.B. DE C.V.', 'SERVICIOS FINANCIEROS'],
  'BBVA.MX': ['BANCO BILBAO VIZCAYA ARGENTARIA, S.A.', 'SERVICIOS FINANCIEROS'],
  'OMAB.MX': ['GRUPO AEROPORTUARIO DEL CENTRO NORTE, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'KOFUBL.MX': ['COCA-COLA FEMSA, S.A.B. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'BBAJIOO.MX': ['BANCO DEL BAJÍO, S.A., INSTITUCIÓN DE BANCA MÚLTIPLE', 'SERVICIOS FINANCIEROS'],
  'MEDICAB.MX': ['MEDICA SUR, S.A.B. DE C.V.', 'SALUD'],
  'AC.MX': ['ARCA CONTINENTAL, S.A.B. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'TS.MX': ['TENARIS S.A.', 'MATERIALES'],
  'GAPB.MX': ['GRUPO AEROPORTUARIO DEL PACIFICO, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'HERDEZ.MX': ['GRUPO HERDEZ, S.A.B. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'CMOCTEZ.MX': ['CORPORACION MOCTEZUMA, S.A.B. DE C.V.', 'MATERIALES'],
  'ASURB.MX': ['GRUPO AEROPORTUARIO DEL SURESTE, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'LABB.MX': ['GENOMMA LAB INTERNACIONAL, S.A.B. DE C.V.', 'SALUD'],
  'VESTA.MX': ['CORPORACIÓN INMOBILIARIA VESTA, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'RA.MX': ['REGIONAL, S.A.B. DE C.V.', 'SERVICIOS FINANCIEROS'],
  'GRUMAB.MX': ['GRUMA, S.A.B. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'ALTERNAB.MX': ['ALTERNA ASESORIA INTERNACIONAL, S.A.B. DE C.V.', 'SERVICIOS FINANCIEROS'],
  'CYDSASAA.MX': ['CYDSA, S.A.B. DE C.V.', 'MATERIALES'],
  'GFNORTEO.MX': ['GRUPO FINANCIERO BANORTE, S.A.B DE C.V.', 'SERVICIOS FINANCIEROS'],
  'GMEXICOB.MX': ['GRUPO MEXICO, S.A.B. DE C.V.', 'MATERIALES'],
  'LIVEPOLC-1.MX': ['EL PUERTO DE LIVERPOOL, S.A.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'VINTE.MX': ['VINTE VIVIENDAS INTEGRALES, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'RLHA.MX': ['RLH PROPERTIES, S.A.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'FEMSAUBD.MX': ['FOMENTO ECONÓMICO MEXICANO, S.A.B. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'DINEB.MX': ['DINE, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'CADUA.MX': ['CORPOVAEL S.A.B. DE C.V.', 'INDUSTRIAL'],
  'SORIANAB.MX': ['ORGANIZACION SORIANA, S.A.B. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'GCC.MX': ['GCC, S.A.B. DE C.V.', 'MATERIALES'],
  'SPORTS.MX': ['GRUPO SPORTS WORLD, S.A.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'INVEXA.MX': ['INVEX CONTROLADORA, S.A.B. DE C.V.', 'SERVICIOS FINANCIEROS'],
  'COLLADO.MX': ['G COLLADO, S.A.B. DE C.V.', 'MATERIALES'],
  'RCENTROA.MX': ['GRUPO RADIO CENTRO, S.A.B. DE C.V.', 'SERVICIOS DE TELECOMUNICACIONES'],
  'IDEALB-1.MX': ['IMPULSORA DEL DESARROLLO Y EL EMPLEO EN AMERICA LATINA, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'PE&OLES.MX': ['INDUSTRIAS PEÑOLES, S. A.B. DE C. V.', 'MATERIALES'],
  'ACCELSAB.MX': ['ACCEL, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'ALSEA.MX': ['ALSEA, S.A.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'GMXT.MX': ['GMÉXICO TRANSPORTES, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'KIMBERA.MX': ['KIMBERLY - CLARK DE MEXICO S.A.B. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'ALFAA.MX': ['ALFA, S.A.B. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'GAVA.MX': ['ACOSTA VERDE, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'LAMOSA.MX': ['GRUPO LAMOSA, S.A.B. DE C.V.', 'MATERIALES'],
  'GPH1.MX': ['GRUPO PALACIO DE HIERRO, S.A.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'TLEVISAA.MX': ['GRUPO TELEVISA, S.A.B.', 'SERVICIOS DE TELECOMUNICACIONES'],
  'GNP.MX': ['GRUPO NACIONAL PROVINCIAL, S.A.B.', 'SERVICIOS FINANCIEROS'],
  'BOLSAA.MX': ['BOLSA MEXICANA DE VALORES, S.A.B. DE C.V.', 'SERVICIOS FINANCIEROS'],
  'ARISTOSA.MX': ['CONSORCIO ARISTOS, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'SAREB.MX': ['SARE HOLDING, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'QBINDUSA.MX': ['Q.B. INDUSTRIAS, S.A. DE C.V.', 'MATERIALES'],
  'AGRIEXPA.MX': ['AGRO INDUSTRIAL EXPORTADORA, S.A. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'SAVIAA.MX': ['SAVIA, S.A. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'CABLECPO.MX': ['EMPRESAS CABLEVISION, S.A. DE C.V.', 'SERVICIOS DE TELECOMUNICACIONES'],
  'QUMMAB.MX': ['GRUPO QUMMA, S.A. DE C.V.', 'SERVICIOS DE TELECOMUNICACIONES'],
  'FINDEP.MX': ['FINANCIERA INDEPENDENCIA, S.A.B. DE C.V. SOFOM, E.N.R.', 'SERVICIOS FINANCIEROS'],
  'LASEG.MX': ['LA LATINOAMERICANA SEGUROS, S.A.', 'SERVICIOS FINANCIEROS'],
  'EDOARDOB.MX': ['EDOARDOS MARTIN, S.A.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'GOMO.MX': ['GRUPO COMERCIAL GOMO, S.A. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'HIMEXSAA.MX': ['HIMEXSA, S.A.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'IASASA.MX': ['INDUSTRIA AUTOMOTRIZ, S.A. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'VASCONI.MX': ['GRUPO VASCONIA S.A.B.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'HOMEX.MX': ['DESARROLLADORA HOMEX, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'CREAL.MX': ['CREDITO REAL, S.A.B. DE C.V., SOFOM, E.N.R.', 'SERVICIOS FINANCIEROS'],
  'TEAKCPO.MX': ['PROTEAK UNO, S.A.B. DE C.V.', 'MATERIALES'],
  'GFAMSAA.MX': ['GRUPO FAMSA, S.A.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'LASITE.MX': ['SITIOS LATINOAMÉRICA, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'ELEKTRA.MX': ['GRUPO ELEKTRA, S.A.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'AXTELCPO.MX': ['AXTEL, S.A.B. DE C.V.', 'SERVICIOS DE TELECOMUNICACIONES'],
  'ORBIA.MX': ['ORBIA ADVANCE CORPORATION, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'UNIFINA.MX': ['UNIFIN FINANCIERA, S.A.B. DE C.V.', 'SERVICIOS FINANCIEROS'],
  'AGUILASCPO.MX': ['OLLAMANI, S.A.B', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'VITROA.MX': ['VITRO, S.A.B. DE C.V.', 'MATERIALES'],
  'NEMAKA.MX': ['NEMAK, S.A.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'VOLARA.MX': ['CONTROLADORA VUELA COMPAÑÍA DE AVIACIÓN, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'AZTECACPO.MX': ['TV AZTECA, S.A.B. DE C.V.', 'SERVICIOS DE TELECOMUNICACIONES'],
  'CUERVO.MX': ['BECLE, S.A.B. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'SITES1A-1.MX': ['OPERADORA DE SITES MEXICANOS, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'TRAXIONA.MX': ['GRUPO TRAXIÓN S.A.B DE C.V.', 'INDUSTRIAL'],
  'CONVERA.MX': ['CONVERTIDORA INDUSTRIAL, S.A.B. DE C.V.', 'MATERIALES'],
  'AUTLANB.MX': ['COMPAÑIA MINERA AUTLAN, S.A.B. DE C. V.', 'MATERIALES'],
  'ALPEKA.MX': ['ALPEK, S.A.B. DE C.V.', 'MATERIALES'],
  'AGUA.MX': ['GRUPO ROTOPLAS, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'MEGACPO.MX': ['MEGACABLE HOLDINGS, S.A.B. DE C.V.', 'SERVICIOS DE TELECOMUNICACIONES'],
  'PV.MX': ['PEÑA VERDE S.A.B.', 'SERVICIOS FINANCIEROS'],
  'PROCORPB.MX': ['PROCORP, S.A.B. DE C.V.', 'SERVICIOS FINANCIEROS'],
  'FRES.MX': ['FRESNILLO PLC', 'MATERIALES'],
  'GISSAA.MX': ['GRUPO INDUSTRIAL SALTILLO, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'GMD.MX': ['GRUPO MEXICANO DE DESARROLLO, S.A.B.', 'INDUSTRIAL'],
  'AMXB.MX': ['AMERICA MOVIL, S.A.B. DE C.V.', 'SERVICIOS DE TELECOMUNICACIONES'],
  'WALMEX.MX': ['WAL - MART DE MEXICO, S.A.B. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'GBMAAABM.MX': ['CORPORATIVO GBM, S.A.B. DE C. V.', 'SERVICIOS FINANCIEROS'],
  'ARA.MX': ['CONSORCIO ARA, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'VALUEGFO.MX': ['VALUE GRUPO FINANCIERO, S.A.B. DE C.V.', 'SERVICIOS FINANCIEROS'],
  'POCHTECB.MX': ['GRUPO POCHTECA, S.A.B. DE C.V.', 'MATERIALES'],
  'CEMEXCPO.MX': ['CEMEX, S.A.B. DE C.V.', 'MATERIALES'],
  'ANB.MX': ['ANHEUSER-BUSCH INBEV SA/NV', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'HOTEL.MX': ['GRUPO HOTELERO SANTA FE, S.A.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'POSADASA.MX': ['GRUPO POSADAS, S.A.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'PLANI.MX': ['PLANIGRUPO LATAM, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'CIDMEGA.MX': ['GRUPE, S.A.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'GFMULTIO.MX': ['GRUPO FINANCIERO MULTIVA S.A.B. DE C.V.', 'SERVICIOS FINANCIEROS'],
  'CMRB.MX': ['CMR, S.A.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'GICSAB.MX': ['GRUPO GICSA, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'BIMBOA.MX': ['GRUPO BIMBO, S.A.B. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'LACOMERUBC.MX': ['LA COMER S.A.B. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'HCITY.MX': ['PROMOTORA DE HOTELES NORTE 19, S.A.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS'],
  'KUOA.MX': ['KUO, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'FINAMEXO.MX': ['CASA DE BOLSA FINAMEX, S.A.B. DE C.V.', 'SERVICIOS FINANCIEROS'],
  'ICHB.MX': ['INDUSTRIAS CH, S.A.B. DE C.V.', 'MATERIALES'],
  'GIGANTE.MX': ['GRUPO GIGANTE, S.A.B. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'MFRISCOA-1.MX': ['MINERA FRISCO, S.A.B. DE C.V.', 'MATERIALES'],
  'SIMECB.MX': ['GRUPO SIMEC, S.A.B. DE C.V.', 'MATERIALES'],
  'CULTIBAB.MX': ['ORGANIZACIÓN CULTIBA, S.A.B. DE CV', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'ICA.MX': ['EMPRESAS ICA, S.A.B. DE C.V.', 'INDUSTRIAL'],
  'MINSAB.MX': ['GRUPO MINSA, S.A.B. DE C.V.', 'PRODUCTOS DE CONSUMO FRECUENTE'],
  'DIABLOSO.MX': ['DIABLOS ROJOS DEL MEXICO, S.A.P.I.B. DE C.V.', 'SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS']
}

data = []
for ticker, info in empresas_bmv.items():
    data.append({
        'Clave Emisora': ticker,
        'Razon Social': info[0],
        'Sector': info[1]
    })

EMPRESAS_BMV = pd.DataFrame(data)
EMPRESAS_BMV

,Clave Emisora,Razon Social,Sector
0,VISTAA.MX,"VISTA ENERGY, S.A.B. DE C.V.",ENERGÍA
1,CIEB.MX,"CORPORACION INTERAMERICANA DE ENTRETENIMIENTO,...",SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS
2,CHDRAUIB.MX,"GRUPO COMERCIAL CHEDRAUI, S.A.B. DE C.V.",PRODUCTOS DE CONSUMO FRECUENTE
3,CTAXTELA.MX,"CONTROLADORA AXTEL, S.A.B. DE C.V.",SERVICIOS DE TELECOMUNICACIONES
4,PASAB.MX,"PROMOTORA AMBIENTAL, S.A.B. DE C.V.",INDUSTRIAL
...,...,...,...
123,SIMECB.MX,"GRUPO SIMEC, S.A.B. DE C.V.",MATERIALES
124,CULTIBAB.MX,"ORGANIZACIÓN CULTIBA, S.A.B. DE CV",PRODUCTOS DE CONSUMO FRECUENTE
125,ICA.MX,"EMPRESAS ICA, S.A.B. DE C.V.",INDUSTRIAL
126,MINSAB.MX,"GRUPO MINSA, S.A.B. DE C.V.",PRODUCTOS DE CONSUMO FRECUENTE


##**RENDIMIENTOS Y VARIANZAS DE LAS ACCIONES**

In [103]:
acciones_analizadas = []
rendimientos = []
varianzas = []

fecha_inicial = '2025-01-01'
fecha_final = '2026-01-01'

for accion in EMPRESAS_BMV['Clave Emisora']:
    precios_accion = yf.Ticker(accion).history(start = fecha_inicial, end = fecha_final)['Close']
    precios_accion.dropna(inplace=True)

    rendimiento = precios_accion.pct_change().dropna().mean()
    varianza = precios_accion.pct_change().dropna().var()

    acciones_analizadas.append(accion)
    rendimientos.append(rendimiento)
    varianzas.append(varianza)

RESULTADOS = pd.DataFrame({'Clave Emisora': acciones_analizadas,
                           'Rendimiento': rendimientos,
                           'Varianza': varianzas})

EMPRESAS_BMV = EMPRESAS_BMV.merge(RESULTADOS, left_on = 'Clave Emisora', right_on = 'Clave Emisora')
EMPRESAS_BMV

ERROR:yfinance:$GMXT.MX: possibly delisted; no timezone found
ERROR:yfinance:$ALFAA.MX: possibly delisted; no timezone found


,Clave Emisora,Razon Social,Sector,Rendimiento,Varianza
0,VISTAA.MX,"VISTA ENERGY, S.A.B. DE C.V.",ENERGÍA,-0.000411,0.001331
1,CIEB.MX,"CORPORACION INTERAMERICANA DE ENTRETENIMIENTO,...",SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS,0.002758,0.000536
2,CHDRAUIB.MX,"GRUPO COMERCIAL CHEDRAUI, S.A.B. DE C.V.",PRODUCTOS DE CONSUMO FRECUENTE,0.000133,0.000249
3,CTAXTELA.MX,"CONTROLADORA AXTEL, S.A.B. DE C.V.",SERVICIOS DE TELECOMUNICACIONES,0.004582,0.000918
4,PASAB.MX,"PROMOTORA AMBIENTAL, S.A.B. DE C.V.",INDUSTRIAL,0.000099,0.000063
...,...,...,...,...,...
123,SIMECB.MX,"GRUPO SIMEC, S.A.B. DE C.V.",MATERIALES,-0.000021,0.000068
124,CULTIBAB.MX,"ORGANIZACIÓN CULTIBA, S.A.B. DE CV",PRODUCTOS DE CONSUMO FRECUENTE,0.000594,0.000224
125,ICA.MX,"EMPRESAS ICA, S.A.B. DE C.V.",INDUSTRIAL,0.000000,0.000000
126,MINSAB.MX,"GRUPO MINSA, S.A.B. DE C.V.",PRODUCTOS DE CONSUMO FRECUENTE,0.000514,0.000028


##**ACCIONES PERFIL CONSERVADOR**

In [104]:
varianza_maxima_conservador = CORTES_VOLATILIDADES_PERFILES_DE_RIESGO.loc[CORTES_VOLATILIDADES_PERFILES_DE_RIESGO['Perfiles de Riesgo'] == 'Perfil Conservador', 'Varianza Máxima'].iloc[0]
varianza_minima_conservador = CORTES_VOLATILIDADES_PERFILES_DE_RIESGO.loc[CORTES_VOLATILIDADES_PERFILES_DE_RIESGO['Perfiles de Riesgo'] == 'Perfil Conservador', 'Varianza Mínima'].iloc[0]

EMPRESAS_CONSERVADOR = EMPRESAS_BMV[(EMPRESAS_BMV['Varianza'] >= varianza_minima_conservador) & (EMPRESAS_BMV['Varianza'] < varianza_maxima_conservador)]
#EMPRESAS_CONSERVADOR = EMPRESAS_CONSERVADOR[EMPRESAS_CONSERVADOR['Rendimiento'] > 0]
EMPRESAS_CONSERVADOR

,Clave Emisora,Razon Social,Sector,Rendimiento,Varianza
44,COLLADO.MX,"G COLLADO, S.A.B. DE C.V.",MATERIALES,0.000000,0.000000
45,RCENTROA.MX,"GRUPO RADIO CENTRO, S.A.B. DE C.V.",SERVICIOS DE TELECOMUNICACIONES,0.000000,0.000000
56,TLEVISAA.MX,"GRUPO TELEVISA, S.A.B.",SERVICIOS DE TELECOMUNICACIONES,0.000000,0.000000
59,ARISTOSA.MX,"CONSORCIO ARISTOS, S.A.B. DE C.V.",INDUSTRIAL,0.000000,0.000000
60,SAREB.MX,"SARE HOLDING, S.A.B. DE C.V.",INDUSTRIAL,0.000000,0.000000
61,QBINDUSA.MX,"Q.B. INDUSTRIAS, S.A. DE C.V.",MATERIALES,0.000000,0.000000
62,AGRIEXPA.MX,"AGRO INDUSTRIAL EXPORTADORA, S.A. DE C.V.",PRODUCTOS DE CONSUMO FRECUENTE,0.000000,0.000000
63,SAVIAA.MX,"SAVIA, S.A. DE C.V.",PRODUCTOS DE CONSUMO FRECUENTE,0.000000,0.000000
64,CABLECPO.MX,"EMPRESAS CABLEVISION, S.A. DE C.V.",SERVICIOS DE TELECOMUNICACIONES,0.000000,0.000000
65,QUMMAB.MX,"GRUPO QUMMA, S.A. DE C.V.",SERVICIOS DE TELECOMUNICACIONES,0.000000,0.000000


In [105]:
varianza_maxima_moderado = CORTES_VOLATILIDADES_PERFILES_DE_RIESGO.loc[CORTES_VOLATILIDADES_PERFILES_DE_RIESGO['Perfiles de Riesgo'] == 'Perfil Moderado', 'Varianza Máxima'].iloc[0]
varianza_minima_moderado = CORTES_VOLATILIDADES_PERFILES_DE_RIESGO.loc[CORTES_VOLATILIDADES_PERFILES_DE_RIESGO['Perfiles de Riesgo'] == 'Perfil Moderado', 'Varianza Mínima'].iloc[0]

EMPRESAS_MODERADO = EMPRESAS_BMV[(EMPRESAS_BMV['Varianza'] >= varianza_minima_moderado) & (EMPRESAS_BMV['Varianza'] < varianza_maxima_moderado)]
EMPRESAS_MODERADO

,Clave Emisora,Razon Social,Sector,Rendimiento,Varianza
48,ACCELSAB.MX,"ACCEL, S.A.B. DE C.V.",INDUSTRIAL,-0.000057,0.000004
53,GAVA.MX,"ACOSTA VERDE, S.A.B. DE C.V.",INDUSTRIAL,0.000390,0.000019
55,GPH1.MX,"GRUPO PALACIO DE HIERRO, S.A.B. DE C.V.",SERVICIOS Y BIENES DE CONSUMO NO BÁSICOS,0.000013,0.000018
119,FINAMEXO.MX,"CASA DE BOLSA FINAMEX, S.A.B. DE C.V.",SERVICIOS FINANCIEROS,0.000445,0.000018


In [106]:
varianza_maxima_agresivo = CORTES_VOLATILIDADES_PERFILES_DE_RIESGO.loc[CORTES_VOLATILIDADES_PERFILES_DE_RIESGO['Perfiles de Riesgo'] == 'Perfil Agresivo', 'Varianza Máxima'].iloc[0]
varianza_minima_agresivo = CORTES_VOLATILIDADES_PERFILES_DE_RIESGO.loc[CORTES_VOLATILIDADES_PERFILES_DE_RIESGO['Perfiles de Riesgo'] == 'Perfil Agresivo', 'Varianza Mínima'].iloc[0]

EMPRESAS_AGRESIVO = EMPRESAS_BMV[(EMPRESAS_BMV['Varianza'] >= varianza_minima_agresivo) & (EMPRESAS_BMV['Varianza'] < varianza_maxima_agresivo)]
EMPRESAS_AGRESIVO

,Clave Emisora,Razon Social,Sector,Rendimiento,Varianza
4,PASAB.MX,"PROMOTORA AMBIENTAL, S.A.B. DE C.V.",INDUSTRIAL,0.000099,0.000063
5,BEVIDESB.MX,"FARMACIAS BENAVIDES, S.A.B. DE C.V.",SALUD,0.000690,0.000043
14,GPROFUT.MX,"GRUPO PROFUTURO, S.A.B. DE C.V.",SERVICIOS FINANCIEROS,0.000820,0.000068
35,VINTE.MX,"VINTE VIVIENDAS INTEGRALES, S.A.B. DE C.V.",INDUSTRIAL,0.000067,0.000088
43,INVEXA.MX,"INVEX CONTROLADORA, S.A.B. DE C.V.",SERVICIOS FINANCIEROS,0.000577,0.000020
46,IDEALB-1.MX,IMPULSORA DEL DESARROLLO Y EL EMPLEO EN AMERIC...,INDUSTRIAL,0.000669,0.000074
54,LAMOSA.MX,"GRUPO LAMOSA, S.A.B. DE C.V.",MATERIALES,-0.000346,0.000049
57,GNP.MX,"GRUPO NACIONAL PROVINCIAL, S.A.B.",SERVICIOS FINANCIEROS,0.000446,0.000082
96,PROCORPB.MX,"PROCORP, S.A.B. DE C.V.",SERVICIOS FINANCIEROS,0.000429,0.000023
105,POCHTECB.MX,"GRUPO POCHTECA, S.A.B. DE C.V.",MATERIALES,-0.000033,0.000086


#**IMPLEMENTACIÓN DE MODELOS**

##**CLASE PORTAFOLIO**

In [107]:
class Portafolio:
################################################################################
  """
  Esta función inicializa la clase Portafolio.

  Args:
    acciones (list): Lista de Strings; donde cada String es el Ticker de un activo.

    fecha_Inicio (str): Fecha mínima para recuperar los datos de los
    activos definidos en la lista acciones (YYYY-MM-DD).

    fecha_Fin (str): Fecha máxima en la que se recuperarán los datos de los
    activos definidos en la lista acciones (YYYY-MM-DD).

  """
  def __init__(self, acciones, fecha_Inicio, fecha_Fin):
    self.acciones = acciones
    self.fecha_Inicio = fecha_Inicio
    self.fecha_Fin = fecha_Fin

################################################################################
  """
  Esta función crea el atributo precios.

  Args:
    None

  Returns:
    precios (DataFrame): DataFrame que contiene los precios de los activos del objeto Portafolio.
                         Cada campo es un activo declarado en la lista acciones del método constructor; cada fila corresponde a
                         una fecha (entre la fecha_Inicio y fecha_Fin definida en el método constructor).
                         Cada valor del DataFrame corresponde al precio de ese activo en esa fecha.
  """
  @property
  def precios(self):
    PRECIO_ACTIVOS = yf.download(self.acciones, start = self.fecha_Inicio, end = self.fecha_Fin)['Close']
    PRECIO_ACTIVOS = PRECIO_ACTIVOS.dropna(how = 'all')
    PRECIO_ACTIVOS.rename_axis(None, axis = 1, inplace = True)
    PRECIO_ACTIVOS.reset_index(inplace = True)
    PRECIO_ACTIVOS['Date'] = pd.to_datetime(PRECIO_ACTIVOS['Date']).dt.date

    return PRECIO_ACTIVOS
################################################################################
  """
  Esta función crea el atributo rendimientos esperados.

  Args:
    None

  Returns:
    precios (DataFrame): DataFrame que contiene los rendimientos esperados del Portafolio.
  """
  @property
  def rendimientos_esperados(self):
    RENDIMIENTOS_ESPERADOS = pd.DataFrame(self.precios.set_index('Date').pct_change().mean(), columns=['Rendimiento Esperado'])
    RENDIMIENTOS_ESPERADOS.reset_index(inplace=True)
    RENDIMIENTOS_ESPERADOS.rename(columns = {'index': 'Activo'}, inplace = True)

    return RENDIMIENTOS_ESPERADOS
################################################################################
  """
  Esta función crea la matriz de varianzas-covarianzas.

  Args:
    None

  Returns:
    precios (DataFrame): DataFrame que muestra las varianzas (elementos de la diagonal) y covarianzas
    de los activos del Portafolio.
  """
  @property
  def varianzas_covarianzas(self):
    MATRIZ_COVARIANZAS = self.precios.drop(columns=['Date']).pct_change().cov()

    return MATRIZ_COVARIANZAS
################################################################################
  """
  El método optimizacion_Markowitz, obtiene la composición óptima del objeto portafolio alineado a la toería de Harry Markowitz.

  Args:
    None

  Returns:
    pesos_Markowitz (OrderedDict): Diccionario ordenado con los pesos de cada activo en el portafolio óptimo según Markowitz, redondeado a 2 decimales y normalizado para que la suma sea 1.
  """
  def optimizacion_Markowitz(self, tasa_libre_riesgo, n_portfolios = 1_000_000, rendimientos_esperados = None, varianzas_covarianzas = None):
    if rendimientos_esperados is None:
      rendimientos_esperados = self.rendimientos_esperados
    if varianzas_covarianzas is None:
      varianzas_covarianzas = self.varianzas_covarianzas

    rendimientos = []
    volatilidades = []
    ratios_sharpe = []
    dataframes_asignaciones = []

    for _ in range(n_portfolios):
      pesos = np.random.random(len(self.acciones))
      pesos /= np.sum(pesos)
      ASIGNACIONES = pd.DataFrame(pesos, index = self.acciones, columns = ['Asignacion'])

      rendimiento = np.dot(pesos, rendimientos_esperados['Rendimiento Esperado'].to_numpy())
      volatilidad = np.sqrt(np.dot(pesos.T, np.dot(varianzas_covarianzas, pesos)))

      rendimientos.append(rendimiento)
      volatilidades.append(volatilidad)
      ratios_sharpe.append((rendimiento - tasa_libre_riesgo) / volatilidad)
      dataframes_asignaciones.append(ASIGNACIONES)

    ASIGNACION_MARKOWITZ = dataframes_asignaciones[np.argmax(ratios_sharpe)]
    ASIGNACION_MARKOWITZ = ASIGNACION_MARKOWITZ.reset_index()
    ASIGNACION_MARKOWITZ.columns = ['Ticker', 'Asignacion Markowitz']

    #Redondeamos todo a 4 decimales
    ASIGNACION_MARKOWITZ['Asignacion Markowitz'] = ASIGNACION_MARKOWITZ['Asignacion Markowitz'].round(4)

    #Ajustamos el último valor para que la suma sea exactamente 1.0
    diferencia = 1.0 - ASIGNACION_MARKOWITZ['Asignacion Markowitz'].sum()
    ASIGNACION_MARKOWITZ.iloc[-1, 1] += round(diferencia, 4)

    return ASIGNACION_MARKOWITZ
################################################################################
  """
  El método optimizacion_BlackLitterman optimiza el portafolio utilizando el modelo Black-Litterman, aceptando vistas
  absolutas o relativas y gestionando su incertidumbre con el método de original de BL.
  Devuelve directamente los pesos Black-Litterman calculados.

  Args:
      views_absolutas: (dict, optional): Diccionario de vistas absoluta(ticker: rendimiento esperado).
      Ej: {'META': 0.05}.
      Default a None.

      views_relativas_P: (pd.DataFrame, optional): Matriz P para vistas relativas.
      Las filas representan las vistas, las columnas los tickers.
      Ej: pd.DataFrame([[1, -1, 0]], columns=['AAPL', 'MSFT', 'GOOG'], index=['AAPL_vs_MSFT']).Default a None.

      views_relativas_Q: (pd.Series, optional): Vector Q con los valores de las vistas relativas.El índice debe coincidir con el
      índice de views_relativas_P.
      Ej: pd.Series([0.02], index=['AAPL_vs_MSFT']).
      Default a None.
  """
  def optimizacion_BlackLitterman(self, capitalizacion_activos: dict, tasa_libre_riesgo: float, rendimiento_esperado_mercado: float, varianza_mercado: float,
      varianzas_covarianzas: pd.DataFrame = None, tau: float = 0.05, P: np.array = None, Q: np.array = None, n_portfolios: int = 1_000_000):

    if varianzas_covarianzas is None:
      varianzas_covarianzas = self.varianzas_covarianzas

    CAPITALIZACION_ACTIVOS = pd.DataFrame(list(capitalizacion_activos.items()), columns = ['Ticker', 'Capitalizacion'])

    ponderaciones = CAPITALIZACION_ACTIVOS['Capitalizacion'] / CAPITALIZACION_ACTIVOS['Capitalizacion'].sum()

    PONDERACIONES_CAPITALIZACION = pd.DataFrame({
      'Ticker': CAPITALIZACION_ACTIVOS['Ticker'],
      'Ponderacion': ponderaciones
    })

    aversion_riesgo = (rendimiento_esperado_mercado - tasa_libre_riesgo)/varianza_mercado

    w_mkt = PONDERACIONES_CAPITALIZACION.set_index('Ticker')['Ponderacion']

    rendimientos_equilibrio = aversion_riesgo * varianzas_covarianzas.dot(w_mkt)

    RENDIMIENTOS_EQUILIBRIO = pd.DataFrame(rendimientos_equilibrio, columns=['Rendimiento Equilibrio'])
    RENDIMIENTOS_EQUILIBRIO.reset_index(inplace=True)

    P_T = P.T
    P_SIGMA_P_T = np.dot(P, np.dot(varianzas_covarianzas, P_T))
    diagonal_omega = np.diag(P_SIGMA_P_T)

    omega = np.diag(diagonal_omega * tau)

    sigma = varianzas_covarianzas.to_numpy()
    pi = RENDIMIENTOS_EQUILIBRIO['Rendimiento Equilibrio'].to_numpy().reshape(-1, 1)
    #omega = omega.to_numpy()

    tau_sigma_inversa = np.linalg.inv(tau * sigma)
    omega_inversa = np.linalg.inv(omega)

    #A
    A = tau_sigma_inversa + P_T @ omega_inversa @ P
    A = np.linalg.inv(A)

    #B
    B_1 = tau_sigma_inversa @ pi
    B_2 = P_T @ omega_inversa @ Q
    B = B_1 + B_2


    rendimientos_ajustados = A @ B

    tickers_list = varianzas_covarianzas.index.tolist()
    RENDIMIENTOS_AJUSTADOS = pd.DataFrame(rendimientos_ajustados, index = tickers_list, columns = ['Rendimiento Ajustado'])
    RENDIMIENTOS_AJUSTADOS.reset_index(inplace = True)
    RENDIMIENTOS_AJUSTADOS.columns = ['Ticker', 'Rendimiento Ajustado']

    rendimientos = []
    volatilidades = []
    ratios_sharpe = []
    dataframes_asignaciones = []

    for _ in range(n_portfolios):
      pesos = np.random.random(len(self.acciones))
      pesos /= np.sum(pesos)
      ASIGNACIONES = pd.DataFrame(pesos, index = self.acciones, columns = ['Asignacion'])

      rendimiento = np.dot(pesos, RENDIMIENTOS_AJUSTADOS['Rendimiento Ajustado'].to_numpy())
      volatilidad = np.sqrt(np.dot(pesos.T, np.dot(varianzas_covarianzas, pesos)))

      rendimientos.append(rendimiento)
      volatilidades.append(volatilidad)
      ratios_sharpe.append((rendimiento - tasa_libre_riesgo) / volatilidad)
      dataframes_asignaciones.append(ASIGNACIONES)

    ASIGNACION_BLACK_LITTERMAN = dataframes_asignaciones[np.argmax(ratios_sharpe)]
    ASIGNACION_BLACK_LITTERMAN = ASIGNACION_BLACK_LITTERMAN.reset_index()
    ASIGNACION_BLACK_LITTERMAN.columns = ['Ticker', 'Asignacion Black-Litterman']


    #Redondeamos todo a 4 decimales
    ASIGNACION_BLACK_LITTERMAN['Asignacion Black-Litterman'] = ASIGNACION_BLACK_LITTERMAN['Asignacion Black-Litterman'].round(4)

    #Ajustamos el último valor para que la suma sea exactamente 1.0
    diferencia = 1.0 - ASIGNACION_BLACK_LITTERMAN['Asignacion Black-Litterman'].sum()
    ASIGNACION_BLACK_LITTERMAN.iloc[-1, 1] += round(diferencia, 4)

    return ASIGNACION_BLACK_LITTERMAN

##**CREACION DE PORTAFOLIOS**

In [108]:
acciones_portafolio_conservador = EMPRESAS_CONSERVADOR['Clave Emisora'].to_list()
acciones_portafolio_moderado = EMPRESAS_MODERADO['Clave Emisora'].to_list()
acciones_portafolio_agresivo = EMPRESAS_AGRESIVO['Clave Emisora'].to_list()

fecha_inicial = '2025-01-01'
fecha_final = '2026-01-01'

portafolio_conservador = Portafolio(acciones_portafolio_conservador, fecha_inicial, fecha_final)
portafolio_moderado = Portafolio(acciones_portafolio_moderado, fecha_inicial, fecha_final)
portafolio_agresivo = Portafolio(acciones_portafolio_agresivo, fecha_inicial, fecha_final)

##**CÁLCULO DE VARIABLES DE MERCADO**

###**TASA LIBRE DE RIESGO $r_f$**

In [109]:
token_banxico = '6d1357d410effb5edad575bc9bba8f7e231048d7be672adfbfdd88849f8ca5c9'
id_cete = 'SF60633'

url = f'https://www.banxico.org.mx/SieAPIRest/service/v1/series/{id_cete}/datos/{fecha_inicial}/{fecha_final}'
headers = {'Bmx-Token': token_banxico, 'Accept': 'application/json'}

response = requests.get(url, headers=headers)

CETES_28 = response.json()
CETES_28 = pd.DataFrame(CETES_28['bmx']['series'][0]['datos'])
CETES_28.columns = ['Date', 'Rate']
CETES_28['Date'] = pd.to_datetime(CETES_28['Date'], format='%d/%m/%Y')

CETES_28['Rate'].dropna()
CETES_28['Rate'] = CETES_28['Rate'].astype(float)
CETES_28['Rate'] = CETES_28['Rate']/100

#La tasa que ofrece CETES en cada subasta es anual, por lo que se convierte a una diaria
CETES_28['Rate'] = CETES_28['Rate']/252

#Calculo tasa libre de riesgo
tasa_libre_riesgo = CETES_28['Rate'].mean()
print('Tasa Libre de Riesgo: ', round(tasa_libre_riesgo, 4))

Tasa Libre de Riesgo:  0.0003


###**RENDIMIENTO ESPERADO DEL PORTAFOLIO DE MERCADO $\mathbb{E}[r_M]$**

In [110]:
IPC = yf.Ticker('^MXX').history(start = fecha_inicial, end = fecha_final)['Close']
IPC = IPC.dropna(how = 'all')
IPC = IPC.reset_index()
IPC['Date'] = pd.to_datetime(IPC['Date']).dt.date

rendimiento_esperado_mercado = IPC['Close'].pct_change().mean()
print('Rendimiento Esperado Portafolio de Mercado: ', round(rendimiento_esperado_mercado, 4))

Rendimiento Esperado Portafolio de Mercado:  0.0011


###**VARIANZA DE RENDIMIENTO DE PORTAFOLIO DE MERCADO $\sigma^2_M$**

In [111]:
varianza_mercado = IPC['Close'].pct_change().var()
print('Varianza de Rendimiento de Portafolio de Mercado: ', round(varianza_mercado, 4))

Varianza de Rendimiento de Portafolio de Mercado:  0.0001


##**ASIGNACION DE ACTIVOS DE PORTAFOLIOS**

###**MODERADO**

####**MARKOWITZ**

In [112]:
ASIGNACION_MARKOWITZ_MODERADO = portafolio_moderado.optimizacion_Markowitz(tasa_libre_riesgo = tasa_libre_riesgo, n_portfolios = 100_000, rendimientos_esperados = portafolio_moderado.rendimientos_esperados, varianzas_covarianzas = portafolio_moderado.varianzas_covarianzas)
ASIGNACION_MARKOWITZ_MODERADO

/tmp/ipykernel_60980/1340574902.py:36: FutureWarning: YF.download() has changed argument auto_adjust default to True
  PRECIO_ACTIVOS = yf.download(self.acciones, start = self.fecha_Inicio, end = self.fecha_Fin)['Close']
[*********************100%***********************]  4 of 4 completed
/tmp/ipykernel_60980/1340574902.py:36: FutureWarning: YF.download() has changed argument auto_adjust default to True
  PRECIO_ACTIVOS = yf.download(self.acciones, start = self.fecha_Inicio, end = self.fecha_Fin)['Close']
[*********************100%***********************]  4 of 4 completed


,Ticker,Asignacion Markowitz
0,ACCELSAB.MX,0.003100
1,GAVA.MX,0.613200
2,GPH1.MX,0.381600
3,FINAMEXO.MX,0.002100


####**BLACK-LITTERMAN**

In [113]:
capitalizacion_activos_moderado = {
    'ACCELSAB.MX': 4.19,
    'GAVA.MX': 10.75,
    'GPH1.MX': 15.87,
    'FINAMEXO.MX': 1.97
}

In [114]:
P_moderado = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
])

In [115]:
Q_moderado = np.array([
    [0.10],
    [0.10],
    [0.10],
    [0.10]
])

In [116]:
ASIGNACION_BLACK_LITTERMAN_MODERADO = portafolio_moderado.optimizacion_BlackLitterman(capitalizacion_activos = capitalizacion_activos_moderado, tasa_libre_riesgo = tasa_libre_riesgo, rendimiento_esperado_mercado = rendimiento_esperado_mercado,
                                      varianza_mercado = varianza_mercado, varianzas_covarianzas = portafolio_moderado.varianzas_covarianzas,
                                      tau = 0.05, P = P_moderado, Q = Q_moderado, n_portfolios = 100_000)
ASIGNACION_BLACK_LITTERMAN_MODERADO

/tmp/ipykernel_60980/1340574902.py:36: FutureWarning: YF.download() has changed argument auto_adjust default to True
  PRECIO_ACTIVOS = yf.download(self.acciones, start = self.fecha_Inicio, end = self.fecha_Fin)['Close']
[*********************100%***********************]  4 of 4 completed


,Ticker,Asignacion Black-Litterman
0,ACCELSAB.MX,0.617900
1,GAVA.MX,0.133600
2,GPH1.MX,0.114700
3,FINAMEXO.MX,0.133800


###**AGRESIVO**

####**MARKOWITZ**

In [117]:
ASIGNACION_MARKOWITZ_AGRESIVO = portafolio_agresivo.optimizacion_Markowitz(tasa_libre_riesgo = tasa_libre_riesgo, n_portfolios = 100_000, rendimientos_esperados = portafolio_agresivo.rendimientos_esperados, varianzas_covarianzas = portafolio_agresivo.varianzas_covarianzas)
ASIGNACION_MARKOWITZ_AGRESIVO

/tmp/ipykernel_60980/1340574902.py:36: FutureWarning: YF.download() has changed argument auto_adjust default to True
  PRECIO_ACTIVOS = yf.download(self.acciones, start = self.fecha_Inicio, end = self.fecha_Fin)['Close']
[*********************100%***********************]  15 of 15 completed
/tmp/ipykernel_60980/1340574902.py:36: FutureWarning: YF.download() has changed argument auto_adjust default to True
  PRECIO_ACTIVOS = yf.download(self.acciones, start = self.fecha_Inicio, end = self.fecha_Fin)['Close']
[*********************100%***********************]  15 of 15 completed


,Ticker,Asignacion Markowitz
0,PASAB.MX,0.156500
1,BEVIDESB.MX,0.009800
2,GPROFUT.MX,0.050800
3,VINTE.MX,0.009600
4,INVEXA.MX,0.132000
5,IDEALB-1.MX,0.078300
6,LAMOSA.MX,0.157700
7,GNP.MX,0.009900
8,PROCORPB.MX,0.159400
9,POCHTECB.MX,0.014800


####**BLACK-LITTERMAN**

In [118]:
capitalizacion_activos_agresivo = {
    'PASAB.MX': 5.06,
    'BEVIDESB.MX': 10.63,
    'GPROFUT.MX': 34.21,
    'VINTE.MX': 8.75,
    'INVEXA.MX': 15.50,
    'IDEALB-1.MX': 122.92,
    'LAMOSA.MX': 35.21,
    'GNP.MX': 26.85,
    'PROCORPB.MX': 370.00,
    'POCHTECB.MX': 818.37,
    'POSADASA.MX': 14.38,
    'CIDMEGA.MX': 1.63,
    'GIGANTE.MX': 31.07,
    'SIMECB.MX': 83.78,
    'MINSAB.MX': 3.54
}

In [119]:
P_agresivo = np.array([
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
])

In [120]:
Q_agresivo = np.array([
    [0.10],
    [0.10],
    [0.10],
    [0.10],
    [0.10],
    [0.10],
    [0.10],
    [0.10],
    [0.10],
    [0.10],
    [0.10],
    [0.10],
    [0.10],
    [0.10],
    [0.10]
])

In [121]:
ASIGNACION_BLACK_LITTERMAN_AGRESIVO = portafolio_agresivo.optimizacion_BlackLitterman(capitalizacion_activos = capitalizacion_activos_agresivo, tasa_libre_riesgo = tasa_libre_riesgo, rendimiento_esperado_mercado = rendimiento_esperado_mercado,
                                      varianza_mercado = varianza_mercado, varianzas_covarianzas = portafolio_agresivo.varianzas_covarianzas,
                                      tau = 0.05, P = P_agresivo, Q = Q_agresivo, n_portfolios = 100_000)

ASIGNACION_BLACK_LITTERMAN_AGRESIVO

/tmp/ipykernel_60980/1340574902.py:36: FutureWarning: YF.download() has changed argument auto_adjust default to True
  PRECIO_ACTIVOS = yf.download(self.acciones, start = self.fecha_Inicio, end = self.fecha_Fin)['Close']
[*********************100%***********************]  15 of 15 completed


,Ticker,Asignacion Black-Litterman
0,PASAB.MX,0.100200
1,BEVIDESB.MX,0.094700
2,GPROFUT.MX,0.030300
3,VINTE.MX,0.037900
4,INVEXA.MX,0.065400
5,IDEALB-1.MX,0.037400
6,LAMOSA.MX,0.109200
7,GNP.MX,0.085200
8,PROCORPB.MX,0.106700
9,POCHTECB.MX,0.028300


#**EVALUACIÓN DE PORTAFOLIOS**

##**FUNCIÓN EVALUACIÓN DE PORTAFOLIOS**

In [158]:
def evaluacion_portafolio(portafolio_inversion: pd.DataFrame, fecha_inicio: str, fecha_final: str):
  #calculo de rendimiento del portafolio
  portafolio_inversion.columns = ['Ticker', 'Asignacion']
  activos_portafolio = portafolio_inversion['Ticker'].to_list()

  PRECIO_ACTIVOS = yf.download(activos_portafolio, start = fecha_inicio, end = fecha_final)['Close']
  PRECIO_ACTIVOS = PRECIO_ACTIVOS.dropna(how = 'all')
  PRECIO_ACTIVOS.rename_axis(None, axis = 1, inplace = True)
  PRECIO_ACTIVOS.reset_index(inplace = True)
  PRECIO_ACTIVOS['Date'] = pd.to_datetime(PRECIO_ACTIVOS['Date']).dt.date

  RENDIMIENTOS_ACTIVOS = PRECIO_ACTIVOS.set_index('Date').pct_change()
  RENDIMIENTOS_ACTIVOS = RENDIMIENTOS_ACTIVOS.dropna(how = 'all')

  RENDIMIENTOS_PORTAFOLIO = RENDIMIENTOS_ACTIVOS.dot(portafolio_inversion['Asignacion'].to_numpy())
  RENDIMIENTOS_PORTAFOLIO.name = 'Rendimiento'
  rendimiento_portafolio = RENDIMIENTOS_PORTAFOLIO.sum()

  #calculo de varianza del portafolio
  MATRIZ_COVARIANZAS = PRECIO_ACTIVOS.drop(columns=['Date']).pct_change().cov()
  varianza_portafolio = np.dot(portafolio_inversion['Asignacion'].to_numpy().T, np.dot(MATRIZ_COVARIANZAS, portafolio_inversion['Asignacion'].to_numpy()))
  volatilidad_portafolio = np.sqrt(varianza_portafolio)

  #calculo de indice de sharpe
  indice_sharpe = (rendimiento_portafolio - tasa_libre_riesgo) / volatilidad_portafolio

  #calculo de indice de herfindahl-hirschman
  indice_herfindahl_hirschman = (portafolio_inversion['Asignacion']**2).sum()

  #construccion de dataframe
  metricas_portafolio = {
      'Rendimiento': rendimiento_portafolio,
      'Varianza' : varianza_portafolio,
      'Volatilidad': volatilidad_portafolio,
      'Indice Sharpe': indice_sharpe,
      'Indice Herfindahl-Hirschman': indice_herfindahl_hirschman
  }

  METRICAS_PORTAFOLIO = pd.DataFrame(list(metricas_portafolio.items()), columns=['Métrica', 'Valor'])

  return METRICAS_PORTAFOLIO

##**MODERADO**

In [159]:
evaluacion_portafolio(ASIGNACION_MARKOWITZ_MODERADO, fecha_inicial, fecha_final)

/tmp/ipykernel_60980/3149150811.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  PRECIO_ACTIVOS = yf.download(activos_portafolio, start = fecha_inicio, end = fecha_final)['Close']
[*********************100%***********************]  4 of 4 completed


,Métrica,Valor
0,Rendimiento,0.105392
1,Varianza,0.000009
2,Volatilidad,0.003066
3,Indice Sharpe,34.265307
4,Indice Herfindahl-Hirschman,0.521647


In [160]:
evaluacion_portafolio(ASIGNACION_BLACK_LITTERMAN_MODERADO, fecha_inicial, fecha_final)

/tmp/ipykernel_60980/3149150811.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  PRECIO_ACTIVOS = yf.download(activos_portafolio, start = fecha_inicio, end = fecha_final)['Close']
[*********************100%***********************]  4 of 4 completed


,Métrica,Valor
0,Rendimiento,0.017659
1,Varianza,0.000002
2,Volatilidad,0.001550
3,Indice Sharpe,11.185406
4,Indice Herfindahl-Hirschman,0.430708


##**AGRESIVO**

In [161]:
evaluacion_portafolio(ASIGNACION_MARKOWITZ_AGRESIVO, fecha_inicial, fecha_final)

/tmp/ipykernel_60980/3149150811.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  PRECIO_ACTIVOS = yf.download(activos_portafolio, start = fecha_inicio, end = fecha_final)['Close']
[*********************100%***********************]  15 of 15 completed


,Métrica,Valor
0,Rendimiento,0.130420
1,Varianza,0.000005
2,Volatilidad,0.002167
3,Indice Sharpe,60.030030
4,Indice Herfindahl-Hirschman,0.122239


In [162]:
evaluacion_portafolio(ASIGNACION_BLACK_LITTERMAN_AGRESIVO, fecha_inicial, fecha_final)

/tmp/ipykernel_60980/3149150811.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  PRECIO_ACTIVOS = yf.download(activos_portafolio, start = fecha_inicio, end = fecha_final)['Close']
[*********************100%***********************]  15 of 15 completed


,Métrica,Valor
0,Rendimiento,0.063403
1,Varianza,0.000003
2,Volatilidad,0.001863
3,Indice Sharpe,33.858142
4,Indice Herfindahl-Hirschman,0.081037
